### 07B · Emulador de Hs y Tm02: PCs simuladas (PVAR) → f_Hs y f_Tm (XGB 04 y 04B) → ruido conjunto (TVAR bivariado, 05B) → Hs, Tm02
Mismo esquema que el 07, con el periodo. Solo el generador elegido (`h1bb24`). Las f salen de las mismas PCs con los dos XGB (mismas columnas y desfases); el ruido es el del 05B: VAR(20) bivariado por régimen de f_Hs, pares de innovaciones por decil de f_Hs y mapeo por rangos de cada variable por decil de f_Hs × mes.

Salida: `outputs/emulador_hstm/{generador}/sim_XXX.parquet` con `VHM0`, `VTM02`, `f`, `f_tm` (no toca `outputs/emulador/`). La validación completa va en el 08B; aquí solo una comprobación rápida por bloques de 41 años.

In [1]:
import os, glob, json, pickle, time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from scipy.signal import find_peaks
from scipy.stats import genextreme

from utils.pit.piece_wise_marginal import from_normal

os.makedirs("figuras", exist_ok=True)
TINTA, AZUL, NARANJA, GRIS, FONDO = "#13233A", "#2C6E91", "#C8622A", "#7A8594", "#F5F3EE"
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Inter", "Arial", "DejaVu Sans"],
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA, "text.color": TINTA, "xtick.color": GRIS, "ytick.color": GRIS,
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.spines.top": False, "axes.spines.right": False,
})

#### Configuración

In [2]:
point = "p102"
GENERADORES = ["h1bb24"]
COLOR = {"h1bb24": NARANJA}
N_SIMS = 100                                  # las ya hechas se saltan
PERIODO = ("1941", "2022")                     # 2 bloques de 41 años por simulación: 1941–1981 y 1982–2022
BLOQUE = 41
OUT = "outputs/emulador_hstm"

#### Modelos: XGB de Hs (04) y de Tm02 (04B), ruido conjunto (05B) y transformación PIT

In [3]:
modelos = {}
for v, carpeta in [("hs", f"outputs/xgb_{point}"), ("tm", f"outputs/xgb_{point}_tm")]:
    m = XGBRegressor()
    m.load_model(f"{carpeta}/xgb_final.json")
    modelos[v] = (m, json.load(open(f"{carpeta}/xgb_features.json")))
assert modelos["hs"][1]["columns"] == modelos["tm"][1]["columns"]       # mismas columnas y desfases
lags = modelos["hs"][1]["lags_h"]

ruido = pickle.load(open(f"outputs/ruido/ruido_tar_hstm_{point}.pkl", "rb"))
P, U_REG, coef, innov_dec, bordes, pools = (ruido[k] for k in ["P", "U_REG", "coef", "innov_dec", "bordes", "pools"])

tr_ = pickle.load(open(f"outputs/marginal/transformacion_{point}.pkl", "rb"))
marg, harm_z = tr_["marg"], tr_["harm_z"]

def desde_z(z, var):
    """Serie de Z de una variable ('VHM0' o 'VTM02') → unidades físicas, deshaciendo la armónica, Φ y la marginal."""
    zz = pd.DataFrame({"VHM0": 0.0, "VTM02": 0.0}, index=z.index)
    zz[var] = z                                                          # la armónica trata cada columna por separado
    z0 = harm_z.inverse_transform(zz)[var].clip(-6.36, 6.36)
    return pd.Series(marg[var].ppf(from_normal(z0).values), index=z.index)

def f_desde_pcs(pcs):
    """PCs 6-horarias (30 presión + 10 gradiente reconstruido) → f_Hs y f_Tm horarias, igual que en el 04 y el 04B."""
    X = pd.concat({f"l{l}": pcs.shift(freq=pd.Timedelta(hours=l)) for l in lags}, axis=1).reindex(pcs.index)
    X.columns = [f"{c}_l{l[1:]}" for l, c in X.columns]
    X = X.dropna()[modelos["hs"][1]["columns"]]
    return {v: pd.Series(m.predict(X), index=X.index).resample("h").interpolate() for v, (m, _) in modelos.items()}

def ruido_tar(f_hs, rng, burn=24 * 30):
    """TAR bivariado: coeficientes por régimen de f_Hs, pares (e_hs, e_tm) del decil de f_Hs."""
    g = (f_hs.values >= U_REG).astype(int)
    dec = np.digitize(f_hs.values, bordes)
    g, dec = np.r_[np.full(burn, g[0]), g], np.r_[np.full(burn, dec[0]), dec]
    e = np.empty((len(g), 2))
    for d, pool in innov_dec.items():
        sel = dec == d
        e[sel] = pool[rng.integers(len(pool), size=sel.sum())]
    x = np.zeros((len(g) + P, 2))
    for t in range(len(g)):
        c = coef[g[t]]
        x[t + P] = c[0] + x[t:t + P][::-1].ravel() @ c[1:] + e[t]
    return x[P + burn:]

def mapear(u, f_hs, pool):
    """Mapeo por rangos: dentro de cada clase (decil de f_Hs × mes) el ruido toma los cuantiles de su pool de residuos reales."""
    dec = np.digitize(f_hs.values, bordes)
    mes = f_hs.index.month
    r = np.empty(len(u))
    for (m, d), p in pool.items():
        sel = (mes == m) & (dec == d)
        if sel.sum():
            r[sel] = np.quantile(p, (pd.Series(u[sel]).rank().values - 0.5) / sel.sum())
    return r

def olas_desde_f(f_hs, f_tm, rng):
    x = ruido_tar(f_hs, rng)
    r_hs = mapear(x[:, 0], f_hs, pools["r_hs"])
    r_tm = mapear(x[:, 1], f_hs, pools["r_tm"])                        # misma clase (f_Hs) para las dos variables
    return desde_z(f_hs + r_hs, "VHM0"), desde_z(f_tm + r_tm, "VTM02")

In [4]:
#### Comprobación: las PCs históricas por esta misma cadena dan la f del 04 y del 04B (test 2016–2025)
pcs_hist = pd.read_csv("outputs/pca/pcs_msl_30_6h.csv", index_col=0, parse_dates=True).join(
           pd.read_csv("outputs/grad_rec/pcs_grad_rec_6h.csv", index_col=0, parse_dates=True))
f_hist = {v: s.loc["1941":"2025"] for v, s in f_desde_pcs(pcs_hist).items()}
for v, carpeta in [("hs", f"outputs/xgb_{point}"), ("tm", f"outputs/xgb_{point}_tm")]:
    f_ref = pd.read_csv(f"{carpeta}/residuos_horarios.csv", index_col=0, parse_dates=True)["f"]
    t = f_ref.loc["2016":"2025"].index.intersection(f_hist[v].index)
    print(f"corr f_{v} (aquí) vs f ({carpeta}) en test: {f_hist[v].loc[t].corr(f_ref.loc[t]):.4f}  (≈ 1)")

#### Y la cadena completa con la f histórica reproduce el hindcast del 05B en distribución (una realización)
h1, t1 = olas_desde_f(f_hist["hs"].loc["1985":"2025"], f_hist["tm"].loc["1985":"2025"], np.random.default_rng(0))
print(f"Hs q99 {h1.quantile(0.99):.2f} m | Tm02 mediana {t1.median():.2f} s, q99 {t1.quantile(0.99):.2f} s | corr Hs–Tm02 {h1.corr(t1):.2f}")

corr f_hs (aquí) vs f (outputs/xgb_p102) en test: 1.0000  (≈ 1)
corr f_tm (aquí) vs f (outputs/xgb_p102_tm) en test: 1.0000  (≈ 1)
Hs q99 3.78 m | Tm02 mediana 3.69 s, q99 6.89 s | corr Hs–Tm02 0.79


#### Emulador: Hs y Tm02 horarias de cada simulación (las que existen se saltan)

In [5]:
for g in GENERADORES:
    carpeta = f"{OUT}/{g}"
    os.makedirs(carpeta, exist_ok=True)
    ficheros = sorted(glob.glob(f"outputs/var_6h/simulaciones_{g}/sim_*.parquet"))[:N_SIMS]
    for i, fic in enumerate(ficheros):
        f_out = f"{carpeta}/{os.path.basename(fic)}"
        if os.path.exists(f_out):
            continue
        t0 = time.time()
        fg = os.path.join(f"outputs/grad_rec/simulaciones_{g}", os.path.basename(fic).replace("sim_", "grec_"))
        pcs = pd.read_parquet(fic).astype("float64").join(pd.read_parquet(fg).astype("float64"))
        f = {v: s.loc[PERIODO[0]:PERIODO[1]] for v, s in f_desde_pcs(pcs).items()}
        hs, tm = olas_desde_f(f["hs"], f["tm"], np.random.default_rng(i))
        pd.DataFrame({"VHM0": hs, "VTM02": tm, "f": f["hs"], "f_tm": f["tm"]}).astype("float32").to_parquet(f_out)
        print(f"{g} sim {i:03d} ({time.time() - t0:.0f} s)", flush=True)

h1bb24 sim 075 (4 s)
h1bb24 sim 076 (4 s)
h1bb24 sim 077 (4 s)
h1bb24 sim 078 (4 s)
h1bb24 sim 079 (4 s)
h1bb24 sim 080 (4 s)
h1bb24 sim 081 (8 s)
h1bb24 sim 082 (10 s)
h1bb24 sim 083 (8 s)
h1bb24 sim 084 (5 s)
h1bb24 sim 085 (4 s)
h1bb24 sim 086 (4 s)
h1bb24 sim 087 (4 s)
h1bb24 sim 088 (4 s)
h1bb24 sim 089 (4 s)
h1bb24 sim 090 (4 s)
h1bb24 sim 091 (4 s)
h1bb24 sim 092 (4 s)
h1bb24 sim 093 (4 s)
h1bb24 sim 094 (4 s)
h1bb24 sim 095 (4 s)
h1bb24 sim 096 (4 s)
h1bb24 sim 097 (4 s)
h1bb24 sim 098 (4 s)
h1bb24 sim 099 (4 s)


#### Comprobación rápida en bloques de 41 años (la validación completa, con el periodo, en el 08B)

In [6]:
hs_obs = pd.read_csv(f"inputs/waves_bulk_{point}.csv", index_col=0, parse_dates=True)["VHM0"].loc["1985":"2025"].asfreq("h")
U90, U99 = hs_obs.quantile(0.90), hs_obs.quantile(0.99)
print(f"U90 {U90:.2f} m | U99 {U99:.2f} m")

def est4(ix):
    return np.select([ix.month.isin([12, 1, 2]), ix.month.isin([3, 4, 5]), ix.month.isin([6, 7, 8])],
                     ["invierno", "primavera", "verano"], "otoño")

def picos_temporal(s):
    return s.index[find_peaks(s.fillna(0).values, height=U99, distance=72)[0]]

def duraciones(s, u):
    d = np.diff(np.r_[0, (s.values > u).astype(int), 0])
    return np.flatnonzero(d == -1) - np.flatnonzero(d == 1)

def metricas(s):
    """Estadísticos de un bloque de Hs horaria (observado, una realización del hindcast o un bloque del emulador)."""
    n_dec = s.dropna().index.year.nunique() / 10
    pk = picos_temporal(s)
    am = s.groupby(s.index.year).max().dropna()
    c, loc, sc = genextreme.fit(am.values)
    return {"q99": s.quantile(0.99), "q99.9": s.quantile(0.999),
            "temporales / década": len(pk) / n_dec,
            **{f"  {e}": (est4(pk) == e).sum() / n_dec for e in ["invierno", "primavera", "verano", "otoño"]},
            "duración > U99 [h]": duraciones(s, U99).mean(), "duración > U90 [h]": duraciones(s, U90).mean(),
            "máx. anual media": am.mean(), "máx. anual σ": am.std(),
            "GEV ξ": -c, "GEV σ": sc, **{f"Hs{T}": genextreme.isf(1 / T, c, loc, sc) for T in [2, 10, 50, 100]}}

def bloques(s):
    a0, a1 = int(PERIODO[0]), int(PERIODO[1])
    return [s.loc[str(a):str(a + BLOQUE - 1)] for a in range(a0, a1 - BLOQUE + 2, BLOQUE)]

#### Observado, hindcast (cada realización = un bloque de 41 años) y generadores (2 bloques por simulación)
m_obs = pd.Series(metricas(hs_obs))
hind = pd.read_pickle(f"outputs/ruido/olas_hindcast_{point}.pkl")["VHM0"].astype("float64")     # hindcast del 05B
m_sim = {"hindcast": pd.DataFrame([metricas(hind[c].where(hs_obs.reindex(hind.index).notna())) for c in hind.columns])}
am_sim = {"hindcast": [hind[c].groupby(hind.index.year).max().values for c in hind.columns]}
for g in GENERADORES:
    filas, ams = [], []
    for fic in sorted(glob.glob(f"{OUT}/{g}/sim_*.parquet")):
        hs = pd.read_parquet(fic)["VHM0"].astype("float64")
        for b in bloques(hs):
            filas.append(metricas(b))
            ams.append(b.groupby(b.index.year).max().values)
    m_sim[g], am_sim[g] = pd.DataFrame(filas), ams
    print(g, len(filas), "bloques")

U90 1.91 m | U99 3.73 m
h1bb24 200 bloques


In [7]:
#### Tabla principal: mediana de los bloques / observado, y rango percentil del observado (≈ 50 ideal)
tabla = pd.DataFrame({"observado": m_obs})
for k, m in m_sim.items():
    tabla[f"{k} p50/obs"] = m.median() / m_obs
for k, m in m_sim.items():
    tabla[f"{k} rango obs"] = (m < m_obs).mean() * 100
tabla.round(2)

,observado,hindcast p50/obs,h1bb24 p50/obs,hindcast rango obs,h1bb24 rango obs
q99,3.73,1.01,1.01,6.0,30.5
q99.9,5.30,1.01,1.01,12.0,28.0
temporales / década,62.20,1.08,1.04,6.0,26.5
invierno,32.68,1.04,1.10,24.0,14.0
primavera,12.68,1.19,1.04,4.0,37.5
verano,0.73,1.00,1.00,22.0,37.5
otoño,16.10,1.05,0.92,24.0,74.0
duración > U99 [h],10.36,0.95,0.98,94.0,61.5
duración > U90 [h],16.72,0.96,0.97,100.0,92.5
máx. anual media,5.66,1.01,1.00,20.0,43.5


In [8]:
#### Tm02 y dependencia: observado vs bloques del emulador (mediana y p5–p95)
ob = pd.read_csv(f"inputs/waves_bulk_{point}.csv", index_col=0, parse_dates=True).loc["1985":"2025", ["VHM0", "VTM02"]].dropna()
def m_tm(df):
    h, t = df["VHM0"], df["VTM02"]
    pk = picos_temporal(h)
    return {"Tm02 mediana": t.median(), "Tm02 q99": t.quantile(0.99), "corr Hs–Tm02": h.corr(t),
            "Spearman Hs–Tm02": h.corr(t, method="spearman"), "Tm02 en picos (mediana)": t.reindex(pk).median(),
            "peralte q99": (h / (9.81 * t**2 / (2 * np.pi))).quantile(0.99)}
o = pd.Series(m_tm(ob))
filas = []
for fic in sorted(glob.glob(f"{OUT}/h1bb24/sim_*.parquet")):
    df = pd.read_parquet(fic, columns=["VHM0", "VTM02"]).astype("float64")
    filas += [m_tm(b) for b in bloques(df)]
s = pd.DataFrame(filas)
pd.DataFrame({"observado": o, "p5": s.quantile(0.05), "p50": s.median(), "p95": s.quantile(0.95), "rango obs": (s < o).mean() * 100}).round(3)

,observado,p5,p50,p95,rango obs
Tm02 mediana,3.709,3.679,3.705,3.731,59.5
Tm02 q99,6.875,6.820,6.865,6.905,61.0
corr Hs–Tm02,0.772,0.784,0.789,0.793,0.0
Spearman Hs–Tm02,0.809,0.812,0.816,0.820,0.0
Tm02 en picos (mediana),6.676,6.609,6.711,6.811,31.0
peralte q99,0.071,0.101,0.102,0.103,0.0
